# Beginner Python Notebook: GB-GBN Energy Market Investigation

This notebook covers fundamental Python programming concepts applied to the **GB_GBN SPECIFIC DATASET**, featuring:
1. Variables
2. A list
3. A dictionary
4. A loop
5. A conditional
6. Two small custom functions relevant to the power and energy market analysis

## 1. Variables
Variables are used to store data values. Below, we define basic parameters for our analysis, such as file paths, site identifiers, and threshold values.

In [ ]:
# Defining variables
dataset_filename = "GB_GBN SPECIFIC DATASET.xlsx"
market_zone = "Great Britain (GB_GBN)"
price_threshold = 50.0  # EUR/MWh or GBP/MWh threshold for high electricity prices

print(f"Analyzing Market Zone: {market_zone}")
print(f"Dataset File: {dataset_filename}")
print(f"High Price Threshold: {price_threshold}")

## 2. Lists
Lists store multiple items in a single variable. Here, we list key columns related to load, generation, and market prices that we want to analyze.

In [ ]:
# Creating a list of key column names to investigate
key_columns = [
    "GB_GBN_load_actual_entsoe_transparency",
    "GB_GBN_price_day_ahead",
    "GB_GBN_solar_generation_actual",
    "GB_GBN_wind_generation_actual"
]

print("Columns to investigate:")
for col in key_columns:
    print(f"- {col}")

## 3. Dictionaries
Dictionaries store data in key-value pairs. Below, we map dataset configuration metadata.

In [ ]:
# Creating a dictionary for dataset metadata
dataset_metadata = {
    "country": "United Kingdom",
    "bidding_zone": "GB_GBN",
    "time_resolution": "Hourly",
    "total_records": 50401
}

print("Dataset Metadata Dictionary:")
for key, value in dataset_metadata.items():
    print(f"{key.capitalize()}: {value}")

## 4. Loops
Loops allow us to iterate over sequences. Here, we load our dataset using Pandas and use a `for` loop to compute and print the average value for each key column.

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_excel(dataset_filename, sheet_name='Sheet1')

# Using a loop to calculate and print mean values for each key column
print("Mean values of key market columns:")
for col in key_columns:
    mean_val = df[col].mean()
    print(f"{col}: {mean_val:.2f}")

## 5. Conditionals
Conditionals (`if`, `elif`, `else`) make decisions in code. Here, we check the average day-ahead price against our threshold variable to evaluate market conditions.

In [ ]:
# Conditional check on average day-ahead price
avg_price = df['GB_GBN_price_day_ahead'].mean()

print(f"Overall Average Day-Ahead Price: {avg_price:.2f}")

if avg_price > price_threshold:
    print("Status: High price market environment (Average price exceeds threshold).")
elif avg_price > 30.0:
    print("Status: Moderate price market environment (Average price between 30 and threshold).")
else:
    print(
        "Status: Low price market environment (Average price is below 30).")

## 6. Functions Relevant to the Investigation
We define two custom functions:
1. `calculate_renewable_share`: Computes total renewable generation (solar + wind) relative to actual load.
2. `categorize_load_level`: Classifies electrical load into qualitative tiers for grid stress analysis.

In [ ]:
# Function 1: Calculate renewable generation share
def calculate_renewable_share(solar_gen, wind_gen, load):
    """Calculates the fraction of load met by solar and wind generation."""
    total_renewables = (0.0 if pd.isna(solar_gen) else solar_gen) + (0.0 if pd.isna(wind_gen) else wind_gen)
    if pd.isna(load) or load == 0:
        return 0.0
    return total_renewables / load

# Function 2: Categorize electrical load level
def categorize_load_level(load):
    """Categorizes electricity demand load into Low, Medium, or High."""
    if pd.isna(load):
        return "Unknown"
    elif load < 25000:
        return "Low Load"
    elif load < 38000:
        return "Medium Load"
    else:
        return "High Load"

# Test our functions on the first few rows of the dataset
df['Sample_Renewable_Share'] = df.apply(
    lambda row: calculate_renewable_share(row['GB_GBN_solar_generation_actual'], row['GB_GBN_wind_generation_actual'], row['GB_GBN_load_actual_entsoe_transparency']), 
    axis=1
)
df['Load_Category'] = df['GB_GBN_load_actual_entsoe_transparency'].apply(categorize_load_level)

print(df[['utc_timestamp', 'GB_GBN_load_actual_entsoe_transparency', 'Sample_Renewable_Share', 'Load_Category']].head(10))